# Tome 15, Chapter 9: Proofs a computer can check (Python companion)

A miniature proof assistant for the implicational fragment of Section 6.4: a type checker (Exercise 6.3), the three-valued semantics of Theorem 6.7, a decision procedure for provability, and a program synthesiser that turns provable formulas into $\lambda$-terms, which the type checker then verifies. The computer-science application is **program synthesis by type inhabitation**: by Theorem 6.4, searching for a proof *is* searching for a program of a given type. Exercises first, solutions at the end. Pure Python. (For the Lean listings of the chapter, use the online editor, as in Exercise 6.1.)

## Exercises

**P9.1 (★) A type checker.** Formulas are atoms (strings) or `('->', A, B)`; terms are `('var', i)`, `('lam', A, t)`, `('app', t, u)`. Write `typeof(ctx, t)` following Definition 6.2, and check the terms $k$ and $S$ (Exercise 6.2) and the terms of Exercise 6.5.

**P9.2 (★) Three values.** Evaluate formulas in $H=\{0,\frac12,1\}$ (Definition 6.5) and search all valuations for one that refutes a formula. Refute Peirce's law and $\neg\neg p\to p$; check that $\{0,1\}$ refutes neither.

**P9.3 (★★) Deciding provability.** Implement Dyckhoff's contraction-free sequent calculus for implication (1992), which terminates, and decide provability of Peirce's law, $S$, $K$, and the double negation of Peirce's law with $q=\bot$, $\neg\neg((\neg p\to p)\to p)$. Here $\bot$ is just an atom, as in Exercise 6.5: there is no rule "from $\bot$ deduce anything".

**P9.4 (★★) Program synthesis.** For a provable formula, search for a $\beta$-normal term of that type by iterative deepening, and verify it with the type checker of P9.1. Synthesise terms for $S$, for $\neg\neg\neg p\to\neg p$, for $(p\to q)\to((q\to r)\to(p\to r))$ and for $\neg\neg((\neg p\to p)\to p)$.

**P9.5 (★★) How good is the three-valued test?** Enumerate all formulas in atoms $p,q$ with at most $3$ arrows. Among the classical tautologies, how many are provable, how many are refuted by $H$, and how many are neither? What does this say about Exercise 6.6?

## Solutions

**P9.1.** One case per typing rule; by Lemma 6.3 nothing is ever guessed.

In [1]:
def arrow(a, b): return ('->', a, b)
def show(A):
    return A if isinstance(A, str) else '(' + show(A[1]) + ' → ' + show(A[2]) + ')'
class TypeError_(Exception): pass
def typeof(ctx, t):
    tag = t[0]
    if tag == 'var':
        if not 0 <= t[1] < len(ctx): raise TypeError_('unbound variable')
        return ctx[t[1]]
    if tag == 'lam':
        return arrow(t[1], typeof(ctx + [t[1]], t[2]))
    if tag == 'app':
        f, a = typeof(ctx, t[1]), typeof(ctx, t[2])
        if isinstance(f, str) or f[1] != a: raise TypeError_(f'cannot apply {show(f)} to {show(a)}')
        return f[2]
V = lambda i: ('var', i); L = lambda A, t: ('lam', A, t); A_ = lambda t, u: ('app', t, u)
p, q, r, bot = 'p', 'q', 'r', '⊥'
neg = lambda X: arrow(X, bot)
K = L(p, L(q, V(0)))
S = L(arrow(p, arrow(q, r)), L(arrow(p, q), L(p, A_(A_(V(0), V(2)), A_(V(1), V(2))))))
dn = L(p, L(neg(p), A_(V(1), V(0))))
tn = L(neg(neg(neg(p))), L(p, A_(V(0), L(neg(p), A_(V(2), V(1))))))
for name, t in [('K', K), ('S', S), ('p → ¬¬p', dn), ('¬¬¬p → ¬p', tn)]:
    print(name, ':', show(typeof([], t)))
try: typeof([], A_(K, K))
except TypeError_ as e: print('K K rejected:', e)

K : (p → (q → p))
S : ((p → (q → r)) → ((p → q) → (p → r)))
p → ¬¬p : (p → ((p → ⊥) → ⊥))
¬¬¬p → ¬p : ((((p → ⊥) → ⊥) → ⊥) → (p → ⊥))
K K rejected: cannot apply (p → (q → p)) to (p → (q → p))


Variables are numbered by position in the context, as in the convention of Definition 6.2: under $\lambda x_1.\lambda x_2.\lambda x_3$ the variable `V(0)` is $x_1$. And `K K` is rejected: $K$ wants a $p$, not a function.

**P9.2.**

In [2]:
from itertools import product
from fractions import Fraction
half = Fraction(1, 2)
def imp(a, b): return 1 if a <= b else b
def val(A, v):
    if isinstance(A, str): return 0 if A == bot else v[A]
    return imp(val(A[1], v), val(A[2], v))
def atoms(A):
    return set() if A == bot else {A} if isinstance(A, str) else atoms(A[1]) | atoms(A[2])
def refute(A, H):
    at = sorted(atoms(A))
    for vals in product(H, repeat=len(at)):
        v = dict(zip(at, vals))
        if val(A, v) < 1: return v
    return None
peirce = arrow(arrow(arrow(p, q), p), p)
dne = arrow(neg(neg(p)), p)
peirce_bot = arrow(arrow(neg(p), p), p)   # Peirce's law with q = ⊥
for name, F in [('Peirce', peirce), ('¬¬p → p', dne)]:
    print(name, '| refuted in {0,1/2,1} by', refute(F, [0, half, 1]), '| in {0,1}:', refute(F, [0, 1]))

Peirce | refuted in {0,1/2,1} by {'p': Fraction(1, 2), 'q': 0} | in {0,1}: None
¬¬p → p | refuted in {0,1/2,1} by {'p': Fraction(1, 2)} | in {0,1}: None


Exactly the valuations of Corollary 6.8 and Exercise 6.5, and $\{0,1\}$ refutes nothing classical.

**P9.3.** Dyckhoff's calculus LJT, restricted to implication: to prove $G$ from a multiset $\Gamma$, (1) succeed if $G$ is an atom in $\Gamma$; (2) if $G=A\to B$, prove $B$ from $\Gamma,A$; (3) if $\Gamma$ contains an atom $a$ and $a\to B$, replace $a\to B$ by $B$; (4) for $(C\to D)\to B$ in $\Gamma$, with $\Gamma'$ the rest: prove $C\to D$ from $\Gamma',D\to B$ and $G$ from $\Gamma',B$. Every rule decreases a well-founded weight, so the search terminates; Dyckhoff proved it complete for intuitionistic logic. (We do not prove that here; P9.4 cross-checks it.)

In [3]:
from functools import lru_cache
def key(G): return tuple(sorted(G, key=repr))
@lru_cache(maxsize=None)
def prove(gamma, goal):
    G = list(gamma)
    if goal in G: return True
    if not isinstance(goal, str):
        return prove(key(G + [goal[1]]), goal[2])
    for i, F in enumerate(G):
        if isinstance(F, str): continue
        rest = G[:i] + G[i+1:]
        a, B = F[1], F[2]
        if isinstance(a, str):
            if a in rest and prove(key(rest + [B]), goal): return True
        else:
            C, D = a[1], a[2]
            if prove(key(rest + [arrow(D, B)]), arrow(C, D)) and prove(key(rest + [B]), goal): return True
    return False
tests = {'Peirce': peirce, 'K': arrow(p, arrow(q, p)), 'S': typeof([], S),
         '¬¬Peirce⊥': neg(neg(peirce_bot)), '¬¬p → p': dne}
for name, F in tests.items(): print(f'{name:22s} provable: {prove((), F)}')

Peirce                 provable: False
K                      provable: True
S                      provable: True
¬¬Peirce⊥              provable: True
¬¬p → p                provable: False


Peirce's law is not provable, as Corollary 6.8 says, but the double negation of its instance $q=\bot$ is: the logic cannot prove the law, yet it cannot refute it either. (Once the rule "from $\bot$ deduce anything" is added, Glivenko's theorem of 1929 says that a propositional formula is a classical tautology if and only if its double negation is provable intuitionistically.)

**P9.4.** Search for normal terms: a normal term of type $A\to B$ is an abstraction; a normal term of atomic type $a$ is a variable $x:C_1\to\dots\to C_m\to a$ applied to normal terms of types $C_1,\dots,C_m$. Iterative deepening on the depth finds a smallest one.

In [4]:
def args_and_target(T):
    args = []
    while not isinstance(T, str): args.append(T[1]); T = T[2]
    return args, T
def synth(ctx, T, depth):
    if depth == 0: return None
    if not isinstance(T, str):
        body = synth(ctx + [T[1]], T[2], depth)
        return None if body is None else L(T[1], body)
    for i in range(len(ctx) - 1, -1, -1):
        args, tgt = args_and_target(ctx[i])
        if tgt != T: continue
        term, ok = V(i), True
        for Ci in args:
            s = synth(ctx, Ci, depth - 1)
            if s is None: ok = False; break
            term = A_(term, s)
        if ok: return term
    return None
def find(T, maxdepth=8):
    for d in range(1, maxdepth + 1):
        t = synth([], T, d)
        if t is not None: return t, d
    return None, None
names = ['x', 'y', 'z', 'w', 'u', 'v', 's']
def pretty(t, k=0):
    if t[0] == 'var': return names[t[1]]
    if t[0] == 'lam': return 'λ' + names[k] + '.' + pretty(t[2], k + 1)
    return '(' + pretty(t[1], k) + ' ' + pretty(t[2], k) + ')'
goals = {'S': typeof([], S), '¬¬¬p → ¬p': arrow(neg(neg(neg(p))), neg(p)),
         '(p→q)→(q→r)→p→r': arrow(arrow(p, q), arrow(arrow(q, r), arrow(p, r))),
         '¬¬Peirce⊥': neg(neg(peirce_bot))}
for name, T in goals.items():
    t, d = find(T)
    print(f'{name:22s} depth {d}: {pretty(t)}   type checks: {typeof([], t) == T}')
print('Peirce:', find(peirce))

S                      depth 3: λx.λy.λz.((x z) (y z))   type checks: True
¬¬¬p → ¬p              depth 3: λx.λy.(x λz.(z y))   type checks: True
(p→q)→(q→r)→p→r        depth 3: λx.λy.λz.(y (x z))   type checks: True
¬¬Peirce⊥              depth 4: λx.(x λy.(y λz.(x λw.z)))   type checks: True
Peirce: (None, None)


Every synthesised term passes the type checker, so by Theorem 6.4 it *is* a proof. Read the program for $\neg\neg((\neg p\to p)\to p)$: given $k$, a refutation of the law, it builds a proof of the law that, as soon as it is handed an $x:p$, calls $k$ with the constant function returning $x$. A program that "changes its mind" by calling its continuation is exactly how classical reasoning is simulated by intuitionistic programs. Peirce's law itself has no term at any depth we tried, as Corollary 6.8 guarantees for all depths.

**P9.5.**

In [5]:
def formulas(n):
    # formulas in p, q with exactly n arrows
    if n == 0: return [p, q]
    out = []
    for k in range(n):
        for A in formulas(k):
            for B in formulas(n - 1 - k): out.append(arrow(A, B))
    return out
rows = []
for n in range(1, 4):
    taut = [F for F in formulas(n) if refute(F, [0, 1]) is None]
    prov = sum(prove((), F) for F in taut)
    ref3 = sum(refute(F, [0, half, 1]) is not None for F in taut)
    rows.append((n, len(formulas(n)), len(taut), prov, ref3, len(taut) - prov - ref3))
print('arrows  formulas  tautologies  provable  refuted-by-H  neither')
for row in rows: print('%6d %9d %12d %9d %13d %8d' % row)

arrows  formulas  tautologies  provable  refuted-by-H  neither
     1         4            2         2             0        0
     2        16            6         6             0        0
     3        80           40        38             2        0


Each classical tautology here is either provable or refuted by $H$ (the last column), and no formula is both, as soundness demands. So for these small formulas the three-valued algebra suffices; for larger ones it does not (Gödel showed in 1932 that no finite Heyting algebra is complete for all formulas), which is why Exercise 6.6 speaks of *some* finite algebra.